# One pipeline, many calls: a Bundle of traces

When a pipeline calls a model more than once (a draft and its critique, several votes, a tool
call), each call gets its own trace, and the traces go into one `tl.Bundle`. The bundle records
how the calls connect, down to which tokens of a later prompt came from which tokens of an
earlier output. This notebook builds a two-call pipeline, reads those token links, follows an
edit made in the first call into the second, and draws the pipeline as a graph of calls.

The pipeline: GPT-2 small drafts the end of a sentence, then DistilGPT2 reads the draft and says
whether it is true. Both are small public checkpoints (about 900 MB together, downloaded on
first run at pinned revisions); everything runs on a laptop CPU in about a minute.

Reference: [`docs/reference/harness_bundles.md`](../../docs/reference/harness_bundles.md).

In [ ]:
# tqdm warns at import when ipywidgets is missing; this notebook turns progress bars off anyway.
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

import torchlens as tl

hf_logging.disable_progress_bar()
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)

GPT2 = (
    "openai-community/gpt2",
    "607a30d783dfa663caf39e06633721c8d4cfcd7e",
)  # pragma: allowlist secret
DISTILGPT2 = (
    "distilbert/distilgpt2",
    "2290a62682d06624634c1f46a6ad5be0f47f38aa",
)  # pragma: allowlist secret


def load(model_id: str, revision: str) -> torch.nn.Module:
    return AutoModelForCausalLM.from_pretrained(
        model_id, revision=revision, attn_implementation="eager"
    ).eval()


drafter, critic = load(*GPT2), load(*DISTILGPT2)
tokenizer = AutoTokenizer.from_pretrained(GPT2[0], revision=GPT2[1])  # shared by both models

## The pipeline, with each call traced

The pipeline is ordinary Python: generate a draft, decode it, paste it into the critic's prompt,
generate the critique. Two TorchLens lines are added per call:

- `tl.trace(model, tokens)` captures one forward pass over the call's prompt plus the tokens it
  generated.
- A small binding says which positions of that token sequence are the call's own output (its
  `completion`), so TorchLens can later look for that output inside another call's prompt.

After both calls, `bundle.relate("fed_by", "critique", "draft", token_spans="auto")` records
that the critique's prompt contains the draft's output, and asks TorchLens to find exactly where,
by matching token ids.

`steer` is used further down; leave it as `None` for now.

In [ ]:
DRAFT_PROMPT = "The Eiffel Tower is in the city of"
CRITIC_TEMPLATE = "Statement:{draft}\nIs the statement true or false? Answer:"


def greedy(model, ids: torch.Tensor, n_new: int) -> torch.Tensor:
    """Greedy decoding without a KV cache; `model` may be a steered model."""
    with torch.no_grad():
        for _ in range(n_new):
            logits = model(ids, use_cache=False).logits
            ids = torch.cat([ids, logits[:, -1].argmax(dim=-1, keepdim=True)], dim=1)
    return ids


def draft_and_critique(steer=None):
    """Run the pipeline once; return its Bundle and each call's prompt length."""
    bundle, prompt_lengths = tl.Bundle(), {}

    def call(name, model, prompt_text, n_new, spec=None):
        prompt = torch.tensor([tokenizer.encode(prompt_text)])
        tokens = greedy(spec.bind(model) if spec else model, prompt, n_new)
        trace = tl.trace(model, tokens, input_kwargs={"use_cache": False}, intervene=spec)
        output_range = [prompt.shape[1], tokens.shape[1]]  # this call's own tokens
        binding = {"stream": "input", "batch_row": 0, "completion": output_range}
        tl.io.attach_sidecar(trace, "torchlens.boundary_facts", binding)
        bundle.add(trace, names=name)
        prompt_lengths[name] = prompt.shape[1]
        return tokenizer.decode(tokens[0, prompt.shape[1] :])

    draft = call("draft", drafter, DRAFT_PROMPT, 12, steer)
    call("critique", critic, CRITIC_TEMPLATE.format(draft=draft), 8)
    bundle.relate("fed_by", "critique", "draft", token_spans="auto", label="quotes the draft")
    return bundle, prompt_lengths


def token_ids(bundle, name):
    """The token ids a call's trace kept (the prompt plus the generated tokens)."""
    return bundle[name].input_ops[0].out[0].tolist()


b, prompt_lengths = draft_and_critique()
for name in b.names:
    ids = token_ids(b, name)
    cut = prompt_lengths[name]
    print(f"{name:9} prompt: {tokenizer.decode(ids[:cut])!r}")
    print(f"{'':9} output: {tokenizer.decode(ids[cut:])!r}")

## Which tokens of the critique's prompt came from the draft

The `fed_by` row is in `b.relations`. Its `token_spans` are four integers each, half-open:
draft start, draft end, critique start, critique end. `basis="matched"` means TorchLens found the
spans itself and checked that the token ids agree position by position; `coverage` is the share
of the draft's output tokens that were found in the critique's prompt.

In [ ]:
row = b.relations[0]
print(f"{row.from_member} fed_by {row.to_member}")
print(f"basis={row.params['basis']}  coverage={row.params['coverage']:.2f}")

draft_ids, critique_ids = token_ids(b, "draft"), token_ids(b, "critique")
for d0, d1, c0, c1 in row.params["token_spans"]:
    print(f"draft tokens {d0}:{d1}     {tokenizer.decode(draft_ids[d0:d1])!r}")
    print(f"critique tokens {c0}:{c1}  {tokenizer.decode(critique_ids[c0:c1])!r}")

`b.token_pairs(reader, source)` turns the checked spans into one-to-one position pairs, which is
what you index activations with. Here is the critique's whole prompt, token by token, with where
each token came from: a draft position, or the critic's fixed template.

In [ ]:
pairs = b.token_pairs("critique", "draft")
from_draft = {reader_pos: source_pos for source_pos, reader_pos in pairs.pairs}
for pos in range(prompt_lengths["critique"]):
    origin = f"draft token {from_draft[pos]}" if pos in from_draft else "template"
    print(f"{pos:3}  {tokenizer.decode([critique_ids[pos]])!r:14} {origin}")

print("\nupstream of critique:", b.upstream("critique"))
print("downstream of draft:", b.downstream("draft"))

## Draw the pipeline as a graph of calls

`b.draw(nodes="members")` draws one node per call and one edge per relation row, from the
earlier call to the later one, labelled with the relation, the free-text `label` given to
`relate`, and on `fed_by` its basis and coverage. The edge is solid because TorchLens checked its token spans; a row whose spans were
only declared is dashed. (The default, `nodes="ops"`, draws the merged operation graph of every
member instead.)

In [ ]:
b.draw(nodes="members")

## Follow an effect from the draft into the critique

Now edit the first call and see what reaches the second. The edit adds a fixed direction to the
output of GPT-2's block-6 MLP at every token while the draft is generated (`tl.steer`), which
changes the draft.

TorchLens does not push an edit through later calls by itself: it never sees the pipeline's
decoding, string formatting or prompt template, so it cannot rebuild them. Instead you run the
same pipeline again with the draft call steered and hand that second run to the first bundle with
`add_alternative`. TorchLens pairs the calls by name, checks that only the declared call was
steered, adds the second run's calls as `cf_draft` and `cf_critique`, and reports for each call
whether its prompt and its output changed, and from which token.

In [ ]:
direction = torch.randn(768, generator=torch.Generator().manual_seed(0))
direction = direction / direction.norm()
steer = tl.when(tl.module("transformer.h.6.mlp"), tl.steer(direction, strength=120.0, dim=-1))

steered_run, steered_lengths = draft_and_critique(steer)
report = b.add_alternative(steered_run, steered=("draft",))
print(report.table())

for name in ("draft", "critique"):
    ids = token_ids(b, "cf_" + name)
    print(f"\ncf_{name} output: {tokenizer.decode(ids[steered_lengths[name] :])!r}")

The draft is `steered`: same prompt, different output. The critique is `absorbed` when its prompt
changed (it quotes the new draft) but its answer did not, and `changed` when the answer changed
too. Either way the critic's internal state moved, and the bundle lets you measure where.

`b.token_divergence` gives the stretches of the critique's prompt that are the same token ids in
both runs: the template text before the quoted draft, and the question after it. Because the
critic is causal, its activations on the shared stretch before the quote must be identical. On the
question after the quote the tokens are identical but the critic has read a different draft, so
any difference there is the edit's effect carried across calls. Below, the critic's final hidden
state (`transformer.ln_f`) is compared token by token on each shared stretch (the change relative
to the original's size), followed by its preference for " true" over " false" at the answer
position. DistilGPT2 is far too small to judge whether a sentence is true; the numbers show that
the edit reached its answer, not that either answer is right.

In [ ]:
d = b.token_divergence("critique", "cf_critique")
print("critique prompts first differ at token", d.prompt_divergence)

hidden = b["critique"]["transformer.ln_f"].out[0]
cf_hidden = b["cf_critique"]["transformer.ln_f"].out[0]
for start, end, cf_start, cf_end in d.shared:
    original = hidden[start:end]
    change = (original - cf_hidden[cf_start:cf_end]).norm(dim=-1) / original.norm(dim=-1)
    text = tokenizer.decode(critique_ids[start:end])
    print(f"tokens {start:2}:{end:2} {text!r:44} largest relative change {change.max():.3f}")

true_id, false_id = tokenizer.encode(" true")[0], tokenizer.encode(" false")[0]
for name, lengths in (("critique", prompt_lengths), ("cf_critique", steered_lengths)):
    answer_pos = lengths["critique"] - 1  # the last prompt token predicts the answer
    logits = b[name]["lm_head"].out[0, answer_pos]
    print(f"{name:12} logit(' true') - logit(' false') = {logits[true_id] - logits[false_id]:+.3f}")

A control: a steer too small to change any drafted token must leave the critique reported
`unchanged`, with the same prompt and the same answer.

In [ ]:
weak = tl.when(tl.module("transformer.h.6.mlp"), tl.steer(direction, strength=1e-3, dim=-1))
control, _ = draft_and_critique()
weak_run, _ = draft_and_critique(weak)
print(control.add_alternative(weak_run, steered=("draft",)).table())

## Both runs in one picture

After `add_alternative` the bundle holds both runs. Each copy is joined to the call it re-ran by
an `alternative_of` row, drawn dashed and without an arrow because the two runs are peers. The
override keeps those rows from deciding the layout (Graphviz `constraint=false`), so each run's
calls stay in one row.

In [ ]:
peers = {
    ("draft", "cf_draft"): {"constraint": "false"},
    ("critique", "cf_critique"): {"constraint": "false"},
}
b.draw(nodes="members", edge_overrides=peers)

## Where to go next

- Voting, routers and tool calls, debates with a judge: the same two steps (add each traced call,
  relate it to the calls it read), with `selected_by` for a decision that picks which call runs.
  Worked versions of each are in
  [`docs/reference/harness_bundles.md`](../../docs/reference/harness_bundles.md).
- Comparing two runs of the same call site by site (`b.compare`, `b.at(site)`) is in
  [`intervention_fork_replay.ipynb`](intervention_fork_replay.ipynb).